# Fetch Data:

In [ ]:
from funcs.process_data_funcs import fetch_data
import pandas as pd
from local_settings import settings
from fredapi import Fred
import os
from funcs.dvc_funcs import run_dvc_command, dagshub_initialization, upload_to_dagshub
from funcs.api_funcs import get_target_arg
from dagshub import get_repo_bucket_client
import sys

def fetch_all_data(target_feature):
    # Initialize FRED API with your API key
    fred = Fred(api_key=settings['api_key'])

    # Fetch and store data for each series in a dictionary
    data_frames = {}
    for series_id in settings['series_ids']:
        frequency = settings['frequency_map'].get(series_id, 'm')  # Default to 'm' if not specified
        try:
            data_frame = fetch_data(series_id, frequency)
            if not data_frame.empty:
                data_frames[series_id] = data_frame
        except Exception as e:
            print(f"Error fetching data for {series_id}: {e}")

    # Combine all data into a single DataFrame
    combined_data = pd.concat(data_frames.values(), axis=1, keys=data_frames.keys())
    combined_data = combined_data.asfreq('MS')
    combined_data.index.name = 'Date'

    # Drop a level from the multi-level columns
    combined_data.columns = combined_data.columns.droplevel(1)

    # Ensure the target feature is included in the dataset
    if target_feature not in combined_data.columns:
        valid_features = ", ".join(combined_data.columns)
        raise ValueError(f"Target feature '{target_feature}' is not available in the dataset. Valid features are: {valid_features}")

    # Save raw data locally
    if not os.path.exists('data/raw'):
        os.makedirs('data/raw')
    combined_data.to_csv('data/raw/raw_data.csv')

    # Upload raw data to Dagshub
    s3 = get_repo_bucket_client("najibabounasr/MacroEconomicAPI")
    s3.upload_file(
        Bucket="MacroEconomicAPI",
        Filename="data/raw/raw_data.csv",
        Key="data/raw/raw_data.csv",
    )

    print("Data fetching complete and tracked with DVC and Dagshub.")
    return combined_data

def main():
    dagshub_initialization()
    if len(sys.argv) < 2:
        raise ValueError("No target feature provided. Please specify the target feature.")
    target_feature = get_target_arg()
    fetch_all_data(target_feature)
    print("Fetch Data Stage Completed")

if __name__ == "__main__":
    main()


# Process Data:

In [ ]:
import os
import pandas as pd
import numpy as np
import configparser
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from funcs.process_data_funcs import (
    impute_missing_values_spline, deflate_nominal_values, apply_log_transformations,
    apply_best_transformations, cap_outliers
)
from funcs.dvc_funcs import run_dvc_command, dagshub_initialization, load_dvc_config, check_remote_config, verify_dvc_remote
from funcs.api_funcs import get_target_arg, get_feature_addition_rounds_arg, get_feature_dropping_threshold_arg, get_tsfresh_fc_params_arg
from dagshub import get_repo_bucket_client
import sys

def main(target):
    # Initialize Dagshub and DVC
    dagshub_initialization()

    # Load combined data from DVC
    combined_data = pd.read_csv('data/raw/raw_data.csv', parse_dates=True, index_col='Date')

    # Perform train/test split
    X = combined_data.drop(columns=[target])
    y = combined_data[[target]]

    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

    # Impute missing values
    quarterly_columns = ['GDP', 'PRFI', 'PNFI', 'EXPGS', 'IMPGS', 'GCE', 'FGCE', 'GDPCTPI']
    treasury_yield_columns = ['DGS2', 'DGS5', 'DGS10']

    for column in quarterly_columns + treasury_yield_columns:
        if column in X_train.columns:
            X_train = impute_missing_values_spline(X_train, column)
            X_test = impute_missing_values_spline(X_test, column)
        elif column in y_train.columns:  # Handle case where target is one of the columns
            y_train = impute_missing_values_spline(y_train, column)
            y_test = impute_missing_values_spline(y_test, column)

    # Name the index column Date
    for df in [X_train, X_test, y_train, y_test]:
        df.index.name = 'Date'

    # Combine X and y dataframes
    train_combined = pd.concat([X_train, y_train], axis=1)
    test_combined = pd.concat([X_test, y_test], axis=1)

    # Ensure CPIAUCSL is included in the dataframes
    cpi_col_name = 'CPIAUCSL'
    columns_to_deflate = ['GDP', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'IMPGS', 'GCE', 'FGCE', 'DSPI']

    # Handle deflation based on conditions
    if target in columns_to_deflate:
        deflated_train = deflate_nominal_values(train_combined[[target, cpi_col_name]], cpi_col_name, [target])
        deflated_test = deflate_nominal_values(test_combined[[target, cpi_col_name]], cpi_col_name, [target])
        train_combined[target] = deflated_train[target]
        test_combined[target] = deflated_test[target]
        columns_to_deflate.remove(target)
    
    train_combined = deflate_nominal_values(train_combined, cpi_col_name, columns_to_deflate)
    test_combined = deflate_nominal_values(test_combined, cpi_col_name, columns_to_deflate)

    # Apply logarithmic transformations
    columns_to_transform = ['GDP', 'PCE', 'PRFI', 'PNFI', 'EXPGS', 'IMPGS', 'GCE', 'FGCE', 'HOUST', 'DSPI']

    if target in columns_to_transform:
        train_combined[target] = apply_log_transformations(train_combined[[target]], [target])
        test_combined[target] = apply_log_transformations(test_combined[[target]], [target])
        columns_to_transform.remove(target)

    train_combined = apply_log_transformations(train_combined, columns_to_transform)
    test_combined = apply_log_transformations(test_combined, columns_to_transform)

    # Separate X and y after transformations
    X_train = train_combined.drop(columns=[target])
    y_train = train_combined[[target]]
    X_test = test_combined.drop(columns=[target])
    y_test = test_combined[[target]]

    # Standardize/Normalize the Data
    scaler = StandardScaler()
    X_train[X_train.columns] = scaler.fit_transform(X_train)
    X_test[X_test.columns] = scaler.transform(X_test)
    y_train[target] = scaler.fit_transform(y_train[target].values.reshape(-1, 1))
    y_test[target] = scaler.transform(y_test[target].values.reshape(-1, 1))

    # Apply Percentage Change
    X_train_pct_change = X_train.pct_change().dropna()
    X_test_pct_change = X_test.pct_change().dropna()
    y_train_pct_change = y_train.pct_change().dropna()
    y_test_pct_change = y_test.pct_change().dropna()

    # Apply the best transformations
    X_train_transformed = apply_best_transformations(X_train_pct_change)
    X_test_transformed = apply_best_transformations(X_test_pct_change)
    y_train_transformed = apply_best_transformations(y_train_pct_change)
    y_test_transformed = apply_best_transformations(y_test_pct_change)

    # Ensure y_train_transformed and y_test_transformed have the correct column name
    y_train_transformed.columns = [target]
    y_test_transformed.columns = [target]

    # Handle outliers in the transformed data
    X_train_transformed = cap_outliers(X_train_transformed, cap_factor=3.0)
    y_train_transformed = cap_outliers(y_train_transformed, cap_factor=3.0)

    # Combine X and y after final transformations
    train_transformed_combined = pd.concat([X_train_transformed, y_train_transformed], axis=1)
    test_transformed_combined = pd.concat([X_test_transformed, y_test_transformed], axis=1)

    # Save the transformed data locally
    if not os.path.exists('data/processed'):
        os.makedirs('data/processed')
    train_transformed_combined.to_csv('data/processed/train_transformed_combined.csv', index=True)
    test_transformed_combined.to_csv('data/processed/test_transformed_combined.csv', index=True)

    # Upload to Dagshub storage
    s3 = get_repo_bucket_client("najibabounasr/MacroEconomicAPI")
    s3.upload_file(
        Bucket="MacroEconomicAPI",  # name of the repo
        Filename="data/processed/train_transformed_combined.csv",  # local path of file to upload
        Key="data/processed/train_transformed_combined.csv",  # remote path where to upload the file
    )
    s3.upload_file(
        Bucket="MacroEconomicAPI",
        Filename="data/processed/test_transformed_combined.csv",
        Key="data/processed/test_transformed_combined.csv",
    )

    # Save individual transformed datasets locally
    X_train_transformed.to_csv('data/processed/X_train_transformed.csv', index=True)
    X_test_transformed.to_csv('data/processed/X_test_transformed.csv', index=True)
    y_train_transformed.to_csv('data/processed/y_train_transformed.csv', index=True)
    y_test_transformed.to_csv('data/processed/y_test_transformed.csv', index=True)

    # Upload individual datasets to Dagshub storage
    s3.upload_file(
        Bucket="MacroEconomicAPI",
        Filename="data/processed/X_train_transformed.csv",
        Key="data/processed/X_train_transformed.csv",
    )
    s3.upload_file(
        Bucket="MacroEconomicAPI",
        Filename="data/processed/X_test_transformed.csv",
        Key="data/processed/X_test_transformed.csv",
    )
    s3.upload_file(
        Bucket="MacroEconomicAPI",
        Filename="data/processed/y_train_transformed.csv",
        Key="data/processed/y_train_transformed.csv",
    )
    s3.upload_file(
        Bucket="MacroEconomicAPI",
        Filename="data/processed/y_test_transformed.csv",
        Key="data/processed/y_test_transformed.csv",
    )

if __name__ == "__main__":
    dagshub_initialization()
    if len(sys.argv) < 2:
        raise ValueError("No target feature provided. Please specify the target feature.")
    target = get_target_arg()
    main(target)
    print("Process Data Stage Completed")


# Engineer Features:

In [ ]:
import os
import random
import sys
import logging
import warnings

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from tsfresh import extract_features
from tsfresh.feature_extraction import MinimalFCParameters, ComprehensiveFCParameters, EfficientFCParameters
from tsfresh.utilities.dataframe_functions import roll_time_series

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from joblib import Parallel, delayed
import optuna

# Custom functions
from funcs.api_funcs import (
    get_feature_addition_rounds_arg, 
    get_target_arg, 
    get_feature_dropping_threshold_arg, 
    get_tsfresh_fc_params_arg
)
from funcs.engineer_features_funcs import (
    compute_mse_scores, 
    optimize_params, 
    evaluate_feature,
    extract_tsfresh_features, 
    compute_baseline_mse, 
    compute_mse_with_added_feature, 
    compute_mse_with_dropped_feature,  
)
from funcs.dvc_funcs import get_repo_bucket_client, dagshub_initialization

# Setup logging
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)

def main(target, feature_addition_rounds, feature_dropping_threshold, fc_parameters, X_train, X_test, y_train, y_test):
    # Verify and initialize DVC remote configuration
    logger.debug("Calling dagshub_initialization()")
    dagshub_initialization()
    logger.debug("dagshub_initialization() called successfully")
    
    np.random.seed(42)
    random.seed(42)
    os.environ['PYTHONHASHSEED'] = str(42)

    # Combine X and y dataframes for feature engineering
    train_combined = pd.concat([X_train, y_train], axis=1)
    test_combined = pd.concat([X_test, y_test], axis=1)

    # Ensure the column names are preserved
    base_features = list(X_train.columns)

    # Suppress Optuna logging
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    # Suppress warnings
    warnings.filterwarnings("ignore", category=DeprecationWarning)
    warnings.filterwarnings("ignore", category=FutureWarning)
    warnings.filterwarnings("ignore", category=UserWarning)
    warnings.filterwarnings("ignore", category=RuntimeWarning)

    # Initial baseline MSE calculation
    logger.debug("Calculating initial baseline MSE")
    baseline_mse_scores, aggregated_baseline_mse, best_params = compute_mse_scores(
        X_train, X_test, y_train, y_test, base_features
    )

    logger.debug(f"Baseline MSE Scores: {baseline_mse_scores}")
    logger.debug(f"Aggregated Baseline MSE Score: {aggregated_baseline_mse}")

    initial_baseline_mse = aggregated_baseline_mse
    initial_mse_xgboost = baseline_mse_scores['XGBoost']
    initial_mse_lightgbm = baseline_mse_scores['LightGBM']

    # Scale the data
    logger.debug("Scaling the data")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    logger.debug("Data scaled successfully")

    # Optimize parameters
    logger.debug("Optimizing XGBoost parameters")
    xgboost_params = optimize_params('XGBoost', X_train_scaled, y_train, X_test_scaled, y_test, n_trials=10)

    logger.debug("Optimizing LightGBM parameters")
    lightgbm_params = optimize_params('LightGBM', X_train_scaled, y_train, X_test_scaled, y_test, n_trials=10)

    # Save as dictionaries to a Python file
    logger.debug("Saving best parameters to a Python file")
    with open('best_params.py', 'w') as f:
        f.write(f"xgboost_params = {xgboost_params}\n")
        f.write(f"lightgbm_params = {lightgbm_params}\n")

    logger.debug("Best parameters saved to best_params.py")

    # Skip TSFRESH feature engineering if feature_addition_rounds is 0
    if feature_addition_rounds > 0:
        # Add ID and time columns required by TSFRESH
        train_combined['id'] = 1
        train_combined['time'] = train_combined.index

        test_combined['id'] = 2
        test_combined['time'] = test_combined.index

        # Extract TSFRESH features once
        logger.debug("Extracting TSFRESH features for train set")
        tsfresh_features_train = extract_tsfresh_features(train_combined, column_id='id', column_sort='time', default_fc_parameters=fc_parameters)
        tsfresh_features_train.index = train_combined.index[:len(tsfresh_features_train)]
        logger.debug("TSFRESH features extracted for train set")

        logger.debug("Extracting TSFRESH features for test set")
        tsfresh_features_test = extract_tsfresh_features(test_combined, column_id='id', column_sort='time', default_fc_parameters=fc_parameters)
        tsfresh_features_test.index = test_combined.index[:len(tsfresh_features_test)]
        logger.debug("TSFRESH features extracted for test set")

        all_added_features = []

        # Initialize the baseline MSE without adding any new feature
        baseline_mse_scores, aggregated_baseline_mse = compute_baseline_mse(
            X_train, X_test, y_train, y_test, base_features
        )
        initial_baseline_mse = aggregated_baseline_mse

        # TSFRESH feature selection rounds with parallel processing
        for round_num in range(feature_addition_rounds):
            logger.debug(f"Round {round_num + 1}/{feature_addition_rounds} of feature engineering")

            # Evaluate adding features in parallel
            results = Parallel(n_jobs=-1)(delayed(evaluate_feature)(
                feature, tsfresh_features_train, tsfresh_features_test, X_train, X_test,
                y_train, y_test, base_features, aggregated_baseline_mse, all_added_features
            ) for feature in tsfresh_features_train.columns)
            aggregated_mse_scores_added = [res for res in results if res is not None]

            # Sort and add top 3 features if they improve the model
            aggregated_mse_scores_added.sort(key=lambda x: x[1])
            top_three_to_add = [f for f in aggregated_mse_scores_added[:3] if f[2] > 0]

            if not top_three_to_add:
                logger.debug("No features improved the model in this round.")
                continue

            for feature, _, improvement, _, _ in top_three_to_add:
                improvement = float(improvement)
                base_features.append(feature)
                all_added_features.append(feature)
                X_train[feature] = tsfresh_features_train[feature]
                X_test[feature] = tsfresh_features_test[feature]
                logger.debug(f"Feature added: {feature}, Improvement: {improvement}")

                # Update baseline MSE after adding each feature
                baseline_mse_scores, aggregated_baseline_mse = compute_mse_with_added_feature(
                    X_train, X_test, y_train, y_test, base_features, feature
                )

        # Calculate overall improvement
        overall_improvement = initial_baseline_mse - aggregated_baseline_mse
        logger.debug(f"\nOverall Improvement in MSE after {feature_addition_rounds} rounds: {overall_improvement}")

        # Use base_features and all_added_features to filter out features that weren't picked
        all_features = list(set(base_features + all_added_features))
        X_train = X_train[all_features]
        X_test = X_test[all_features]

        logger.debug("Feature Addition Complete")

    # Set a higher threshold for improvement (e.g., 0.05% of the initial baseline MSE)
    feature_dropping_threshold = float(feature_dropping_threshold)
    threshold = feature_dropping_threshold * ((baseline_mse_scores['XGBoost'] + baseline_mse_scores['LightGBM']) / 2)

    # Calculate and drop features
    aggregated_mse_scores_dropped = []
    for feature in base_features:
        mse_scores, aggregated_mse = compute_mse_with_dropped_feature(X_train, X_test, y_train, y_test, base_features, feature)
        improvement = aggregated_baseline_mse - aggregated_mse

        improvement_status = "improved" if improvement > threshold else "worsened"
        aggregated_mse_scores_dropped.append((feature, aggregated_mse, improvement, improvement_status, mse_scores))

    # Sort and drop the least impactful features if they result in improvement
    aggregated_mse_scores_dropped.sort(key=lambda x: x[1])
    features_to_drop = [f for f in aggregated_mse_scores_dropped if f[2] > threshold]

    if not features_to_drop:
        print("No features were dropped as they did not improve the model.")
    else:
        for feature, _, improvement, _, _ in features_to_drop:
            base_features.remove(feature)
            print(f"Feature dropped: {feature}, Improvement: {improvement}")

    print("Feature Dropping Completed.")

    # Final baseline MSE calculation
    final_mse_scores, aggregated_final_mse, _ = compute_mse_scores(
        X_train, X_test, y_train, y_test, base_features
    )
    # Store final MSE scores for each model
    final_mse_xgboost = final_mse_scores['XGBoost']
    final_mse_lightgbm = final_mse_scores['LightGBM']

    # Calculate improvements
    improvement_xgboost = initial_mse_xgboost - final_mse_xgboost
    improvement_lightgbm = initial_mse_lightgbm - final_mse_lightgbm

    # Print the results
    print("")
    print("THRESHOLD OF 0.002:")
    print(f"Initial MSE for XGBoost: {initial_mse_xgboost}")
    print(f"Final MSE for XGBoost: {final_mse_xgboost}")
    print(f"Improvement in MSE for XGBoost: {improvement_xgboost}\n")

    print(f"Initial MSE for LightGBM: {initial_mse_lightgbm}")
    print(f"Final MSE for LightGBM: {final_mse_lightgbm}")
    print(f"Improvement in MSE for LightGBM: {improvement_lightgbm}")

    # Print the results
    logger.debug(f"Initial MSE for XGBoost: {initial_mse_xgboost}")
    logger.debug(f"Final MSE for XGBoost: {final_mse_xgboost}")
    logger.debug(f"Improvement in MSE for XGBoost: {improvement_xgboost}\n")

    logger.debug(f"Initial MSE for LightGBM: {initial_mse_lightgbm}")
    logger.debug(f"Final MSE for LightGBM: {final_mse_lightgbm}")
    logger.debug(f"Improvement in MSE for LightGBM: {improvement_lightgbm}")

    # Save Data using S3 buckets and .csv files
    if not os.path.exists('data/engineered'):
        os.makedirs('data/engineered')
    X_train.to_csv('data/engineered/X_train_engineered.csv', index=True)
    X_test.to_csv('data/engineered/X_test_engineered.csv', index=True)
    y_train.to_csv('data/engineered/y_train_engineered.csv', index=True)
    y_test.to_csv('data/engineered/y_test_engineered.csv', index=True)

    logger.debug("Engineered data saved locally.")

    # Upload to Dagshub storage
    logger.debug("Uploading to DAGsHub storage")
    s3 = get_repo_bucket_client("najibabounasr/MacroEconomicAPI")
    s3.upload_file(
        Bucket="MacroEconomicAPI",
        Filename="data/engineered/X_train_engineered.csv",
        Key="data/engineered/X_train_engineered.csv",
    )
    s3.upload_file(
        Bucket="MacroEconomicAPI",
        Filename="data/engineered/X_test_engineered.csv",
        Key="data/engineered/X_test_engineered.csv",
    )
    s3.upload_file(
        Bucket="MacroEconomicAPI",
        Filename="data/engineered/y_train_engineered.csv",
        Key="data/engineered/y_train_engineered.csv",
    )
    s3.upload_file(
        Bucket="MacroEconomicAPI",
        Filename="data/engineered/y_test_engineered.csv",
        Key="data/engineered/y_test_engineered.csv",
    )
    logger.debug("Data uploaded to DAGsHub storage")

if __name__ == '__main__':
    target = get_target_arg()
    feature_addition_rounds = get_feature_addition_rounds_arg()
    feature_dropping_threshold = float(get_feature_dropping_threshold_arg())
    tsfresh_fc_params = str(get_tsfresh_fc_params_arg())
    X_train = pd.read_csv('data/processed/X_train_transformed.csv', index_col='Date', parse_dates=True)
    y_train = pd.read_csv('data/processed/y_train_transformed.csv', index_col='Date', parse_dates=True)
    y_test = pd.read_csv('data/processed/y_test_transformed.csv', index_col='Date', parse_dates=True)
    X_test = pd.read_csv('data/processed/X_test_transformed.csv', index_col='Date', parse_dates=True)

    # Map the string to the appropriate TSFRESH parameter object
    if tsfresh_fc_params == 'MinimalFCParameters':
        fc_parameters = MinimalFCParameters()
    elif tsfresh_fc_params == 'ComprehensiveFCParameters':
        fc_parameters = ComprehensiveFCParameters()
    elif tsfresh_fc_params == 'EfficientFCParameters':
        fc_parameters = EfficientFCParameters()
    else:
        # Default to MinimalFCParameters if the provided value is invalid
        fc_parameters = MinimalFCParameters()
        logger.debug("Invalid TSFRESH feature extraction parameters. Defaulting to 'MinimalFCParameters'.")
        print("Invalid TSFRESH feature extraction parameters. Choose from: 'MinimalFCParameters','EfficientFCParameters','ComprehensiveFCParameters'.")

    logger.debug("Starting feature engineering")
    main(target, feature_addition_rounds, feature_dropping_threshold, fc_parameters, X_train, X_test, y_train, y_test)
    logger.debug("Feature engineering finished")


# Train Models:

In [ ]:
import os
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.metrics import mean_squared_error, mean_absolute_error
from funcs.api_funcs import get_target_arg, get_feature_addition_rounds_arg, get_feature_dropping_threshold_arg, get_tsfresh_fc_params_arg
from sklearn.neighbors import KNeighborsRegressor
from autogluon.tabular import TabularPredictor
from funcs.train_model_funcs import clean_data

def optimize_model(objective_function, model_name):
    study = optuna.create_study(direction='minimize')
    study.optimize(objective_function, n_trials=100)
    return study

def objective_knn(trial, X_train, y_train, X_test, y_test):
    param_grid = {
        'n_neighbors': trial.suggest_int('n_neighbors', 1, 50),
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        'algorithm': trial.suggest_categorical('algorithm', ['auto', 'ball_tree', 'kd_tree', 'brute']),
        'leaf_size': trial.suggest_int('leaf_size', 10, 50),
        'p': trial.suggest_int('p', 1, 2)
    }
    
    with mlflow.start_run(nested=True):
        model = KNeighborsRegressor(**param_grid)
        model.fit(X_train, y_train)

        test_predictions = model.predict(X_test)
        test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
        train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))

        mlflow.log_metrics({'test_rmse': test_rmse, 'train_rmse': train_rmse})
        for param_key, param_value in param_grid.items():
            mlflow.log_param(param_key, param_value)
    return test_rmse

def objective_autogluon(trial, X_train, y_train, X_test, y_test, target):
    param_grid = {
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
        'num_boost_round': trial.suggest_int('num_boost_round', 50, 100),
        'num_leaves': trial.suggest_int('num_leaves', 20, 50),
        'feature_fraction': trial.suggest_uniform('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.5, 1.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100),
        'lambda_l1': trial.suggest_loguniform('lambda_l1', 1e-4, 1e+1),
        'lambda_l2': trial.suggest_loguniform('lambda_l2', 1e-4, 1e+1),
        'verbose': -1
    }
    
    with mlflow.start_run(nested=True):
        train_data = pd.concat([X_train, y_train], axis=1)
        train_data.columns = list(X_train.columns) + [target]
        
        predictor = TabularPredictor(label=target, eval_metric='rmse').fit(
            train_data=train_data,
            hyperparameters={'GBM': param_grid},
            num_bag_folds=5,
            ag_args_fit={'num_gpus': 0, 'num_cpus': 1}
        )
        test_predictions = predictor.predict(X_test)
        train_predictions = predictor.predict(X_train)
        
        test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
        train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))

        mlflow.log_metrics({'test_rmse': test_rmse, 'train_rmse': train_rmse})
        for param_key, param_value in param_grid.items():
            mlflow.log_param(param_key, param_value)
    return test_rmse

def objective_lgb(trial, X_train, y_train, X_test, y_test):
    param_grid = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 10, 100),
        'boosting_type': trial.suggest_categorical('boosting_type', ['gbdt', 'dart']),
        'bagging_fraction': trial.suggest_uniform('bagging_fraction', 0.5, 1.0),
        'feature_fraction': trial.suggest_uniform('feature_fraction', 0.5, 1.0),
        'verbosity': -1
    }
    
    with mlflow.start_run(nested=True):
        model = lgb.LGBMRegressor(**param_grid)
        model.fit(X_train, y_train)

        test_predictions = model.predict(X_test)
        train_predictions = model.predict(X_train)

        test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
        train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))

        mlflow.log_metrics({'test_rmse': test_rmse, 'train_rmse': train_rmse})
        for param_key, param_value in param_grid.items():
            mlflow.log_param(param_key, param_value)
    return test_rmse

def objective_xgb(trial, X_train, y_train, X_test, y_test):
    param_grid = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
        'lambda': trial.suggest_loguniform('lambda', 1e-4, 1e+1),
        'alpha': trial.suggest_loguniform('alpha', 1e-4, 1e+1),
        'verbosity': 0
    }
    
    with mlflow.start_run(nested=True):
        model = xgb.XGBRegressor(**param_grid, verbosity=0)
        model.fit(X_train, y_train)

        test_predictions = model.predict(X_test)
        test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))

        mlflow.log_metrics({'test_rmse': test_rmse})
        for param_key, param_value in param_grid.items():
            mlflow.log_param(param_key, param_value)
    return test_rmse

def objective_catboost(trial, X_train, y_train, X_test, y_test):
    param_grid = {
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'l2_leaf_reg': trial.suggest_loguniform('l2_leaf_reg', 1e-4, 1e+1),
        'border_count': trial.suggest_int('border_count', 1, 255),
        'bagging_temperature': trial.suggest_uniform('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_uniform('random_strength', 0.0, 1.0),
        'verbose': 0
    }
    
    with mlflow.start_run(nested=True):
        model = cb.CatBoostRegressor(**param_grid, verbose=0)
        model.fit(X_train, y_train)

        test_predictions = model.predict(X_test)
        train_predictions = model.predict(X_train)

        test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
        train_rmse = np.sqrt(mean_squared_error(y_train, train_predictions))

        mlflow.log_metrics({'test_rmse': test_rmse, 'train_rmse': train_rmse})
        for param_key, param_value in param_grid.items():
            mlflow.log_param(param_key, param_value)
    return test_rmse

def main(target_feature, feature_addition_rounds, feature_dropping_threshold, fc_parameters, X_train, X_test, y_train, y_test):
    mlflow.set_experiment("Model Optimization")

    print(f"Optimizing KNN...")
    optimize_model(lambda trial: objective_knn(trial, X_train, y_train, X_test, y_test), "KNN")

    print(f"Optimizing AutoGluon...")
    optimize_model(lambda trial: objective_autogluon(trial, X_train, y_train, X_test, y_test, target_feature), "AutoGluon")

    print(f"Optimizing LightGBM...")
    optimize_model(lambda trial: objective_lgb(trial, X_train, y_train, X_test, y_test), "LightGBM")

    print(f"Optimizing XGBoost...")
    optimize_model(lambda trial: objective_xgb(trial, X_train, y_train, X_test, y_test), "XGBoost")

    print(f"Optimizing CatBoost...")
    optimize_model(lambda trial: objective_catboost(trial, X_train, y_train, X_test, y_test), "CatBoost")

if __name__ == "__main__":
    mlflow.set_tracking_uri("https://dagshub.com/najibabounasr/MacroEconomicAPI.mlflow")
    dagshub.init("MacroEconomicAPI", "najibabounasr", mlflow=True)
    os.environ['MLFLOW_TRACKING_USERNAME'] = 'najibabounasr'
    os.environ['MLFLOW_TRACKING_PASSWORD'] = 'fbaccfb8cf4e8d2d195cd05e9a53dbfe32323695'

    X_train = pd.read_csv('data/engineered/X_train_engineered.csv', index_col='Date', parse_dates=True)
    y_train = pd.read_csv('data/engineered/y_train_engineered.csv', index_col='Date', parse_dates=True)
    X_test = pd.read_csv('data/engineered/X_test_engineered.csv', index_col='Date', parse_dates=True)
    y_test = pd.read_csv('data/engineered/y_test_engineered.csv', index_col='Date', parse_dates=True)

    X_train, y_train = clean_data(X_train, y_train)
    X_test, y_test = clean_data(X_test, y_test)

    target_feature = get_target_arg()
    feature_addition_rounds = get_feature_addition_rounds_arg()
    feature_dropping_threshold = float(get_feature_dropping_threshold_arg())
    fc_parameters = get_feature_addition_rounds_arg()

    main(target_feature, feature_addition_rounds, feature_dropping_threshold, fc_parameters, X_train, X_test, y_train, y_test)
